# Stage 2: substrate slabs and terminations
Slabs are cut from the Stage 1 relaxed bulks. The in-plane lattice is fixed by the bulk, and c lies along z.

For each distinct termination, two variants are built:
- **symmetric:** the same termination top and bottom, often non-stoichiometric;
- **stoichiometric:** keeps the bulk formula.

A slab with no z → −z symmetry operation is flagged for the dipole correction (`LDIPOL`, `IDIPOL=3`). The bottom half of each slab is frozen for Stage 3.

**Visual check:** terminations, thickness, vacuum, and polarization direction.

In [ ]:
config_path = '../config/pipeline.yaml'
stage = '02_slabs'
stage_dir = '../../stages/02_slabs'

In [ ]:
import warnings
from pathlib import Path
import numpy as np, pandas as pd, yaml
import matplotlib.pyplot as plt
from IPython.display import display
from pymatgen.core import Structure
from ornl_lotf import slabs, viz
from ornl_lotf.manifest import read_manifest, write_manifest
warnings.filterwarnings('ignore')
config = yaml.safe_load(Path(config_path).read_text())
out = Path(stage_dir); out.mkdir(parents=True, exist_ok=True)
bulk_records = read_manifest(out.parent / '01_bulk_relax')
print('relaxed bulks:', [(r.id, r.status) for r in bulk_records])
pd.set_option('display.width', 250, 'display.max_columns', 40, 'display.max_rows', 200)

## Enumerate slabs

In [ ]:
records, rows = slabs.build_slabs(config, bulk_records, out)
write_manifest(out, records)
table = pd.DataFrame(rows); table.to_csv(out / 'slabs.csv', index=False)
print(f'{len(records)} slabs written'); display(table)

## Counts by substrate and facet

In [ ]:
summary = pd.DataFrame([{'polymorph': r.polymorph, 'facet': ''.join(map(str, r.facet)), 'top': r.termination,
                         'variants': '+'.join(r.extra['variants']), 'dipole': r.asymmetric,
                         'polarization': r.extra['polarization'], 'natoms': r.extra['natoms']} for r in records])
display(summary.pivot_table(index=['polymorph', 'facet', 'polarization'], columns='top', values='natoms', aggfunc='count', fill_value=0))

## Plane profiles
Each slab is shown with z on the vertical axis and the vacuum shaded. Filled markers are relaxed atoms and open markers are frozen.

In [ ]:
def profile(ax, rec):
    s = Structure.from_file(rec.path); z = s.cart_coords[:, 2]
    sd = np.array([v[0] for v in s.site_properties['selective_dynamics']])
    for el in sorted({x.symbol for x in s.species}):
        m = np.array([x.symbol == el for x in s.species])
        col = viz.COLORS.get(el, 'gray')
        ax.scatter(np.full(m.sum(), list(viz.COLORS).index(el) if el in viz.COLORS else 0) + np.random.uniform(-.2, .2, m.sum()),
                   z[m], s=18, facecolors=np.where(sd[m], col, 'none'), edgecolors=col, label=el)
    c = s.lattice.c; ax.axhspan(0, z.min(), color='0.92'); ax.axhspan(z.max(), c, color='0.92')
    ax.set_ylim(0, c); ax.set_xticks([])
    ax.set_title(f"{rec.polymorph} {''.join(map(str, rec.facet))}\n{rec.termination} | {'+'.join(rec.extra['variants'])}"
                 f"\n{rec.extra['formula']}{' | dipole' if rec.asymmetric else ''}", fontsize=7)
n = len(records); cols = 6; fig, axes = plt.subplots(int(np.ceil(n / cols)), cols, figsize=(2.2 * cols, 3.4 * np.ceil(n / cols)))
for ax, rec in zip(axes.flat, records): profile(ax, rec)
for ax in list(axes.flat)[n:]: ax.axis('off')
axes.flat[0].set_ylabel('z (Å)'); fig.tight_layout(); fig.savefig(out / 'slab_profiles.png', dpi=130); plt.show()

## Side views (x–z projection, z up, 2× along a)
This is the reference check: each title is drawn in the same axes as its structure. Open markers are frozen atoms.

In [ ]:
titles = [f"{r.polymorph} {''.join(map(str, r.facet))}\n{r.termination} | {'+'.join(r.extra['variants'])}\n{r.extra['formula']}{' | dipole' if r.asymmetric else ''}" for r in records]
fig = viz.side_views([Structure.from_file(r.path) for r in records], titles, cols=6)
fig.savefig(out / 'slab_sideviews.png', dpi=130); plt.show()

## 3D views (side view along b, 2×2 in-plane)
There is one 3Dmol viewer per substrate, polymorph and facet group, and each panel is labelled inside its own canvas. One viewer per slab (33 in total) would go over the browser's limit of about 16 WebGL contexts, and the browser then shows structures under the wrong labels.

In [ ]:
from itertools import groupby
key = lambda r: (r.substrate, r.polymorph, ''.join(map(str, r.facet)))
for k, grp in groupby(records, key=key):
    grp = list(grp)
    print(' '.join(k), '|', ', '.join(r.id.split('-', 4)[-1] for r in grp))
    viz.show_grid([Structure.from_file(r.path) for r in grp],
                  [f"{r.id.split('-', 4)[-1]} | {r.extra['formula']}" for r in grp], cols=len(grp))